# Get the data

This notebook gets PPMI data from the LONI Image and Data Archive (IDA) with your own account, and checks it. It goes with sections 5 and 7 of the PPMI atlas (`atlas/index.html`):

1. **The data dictionary**: every kind of PPMI data and what it is.
2. **What IDA offers**: every file on IDA's Study Data page, lined up with the dictionary and with what you have.
3. **Choose** the files you want, in SQL.
4. **Download** them through IDA, with the session you signed in with in your browser.
5. **Image searches**, which you run in your browser, and 6. **raw images**, which download with no clicks: every DaTscan and structural MRI series.
7. **Check** what arrived, and 8. **query** it all with SQL, the interface the other notebooks use.

Run it top to bottom on your own machine, and again at any time: it fetches only what is still missing.

PPMI shares its data through IDA. Researchers [apply for access](https://www.ppmi-info.org/access-data-specimens/download-data), accept the data use agreement and download with their own account. The agreement forbids sharing participant-level data and applying AI tools to it, so this notebook prints names and counts only. Everything lands in `data/` in this repository, which git ignores, or in the folder named by the `PPMI_DATA` environment variable.

In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline
import sys
sys.path.append("../code")

import shutil
from pathlib import Path

import data
import ida
import plot

p = data.PPMI(clean=False)
print("Data folder:", p.root.as_posix().replace(Path.home().as_posix(), "~", 1))
for note in p.notes:
    print(note)

## 1. The data dictionary

`dictionary` holds `code/catalog.csv`: every kind of data that Appendix A of the PPMI Data User Guide describes, plus the imaging tables in current downloads. Rows marked "various" stand for groups of similar tables. IDA's own list, in section 2, is the final word, because tables are added between editions of the guide.

IDA's *Download* menu has three parts: **Study Data**, a few hundred CSV tables and documents in the groups below; **Image Collections**, the scans, as DICOM or NIfTI files in zips; and **Genetic Data**, sequencing and genotyping files.

| Kind | One row per | Joins on |
|---|---|---|
| participant table | participant | `PATNO` |
| visit table | participant and visit | `PATNO`, `EVENT_ID` |
| log | dated record, such as a medication | `PATNO` and dates |
| imaging table | scan: metadata or a derived measure | `PATNO`, `EVENT_ID`, `IMAGEID` |
| biospecimen table | sample and test | `PATNO`, `CLINICAL_EVENT` |
| curated table | participant and visit, many sources merged | `PATNO`, `EVENT_ID` |
| images, genetic files, sensor data | file | subject and visit in the file and folder names |
| document | | |

In [ ]:
plot.heatmap(p.sql("""PIVOT (SELECT "group", kind FROM dictionary) ON kind USING count(*) GROUP BY "group" ORDER BY "group" """),
             "Kinds of data in each part of IDA", "datasets")

In [ ]:
p.sql("""SELECT "group", subgroup, dataset, kind, format, keys, project, notes FROM dictionary ORDER BY entry""")

The `project` column marks the four sets this repository uses:

- **core**: the twelve tables that `audit.ipynb` needs.
- **pipeline**: three tables the clean layer also reads when they are present: MoCA, the archived Invicro DaTscan ratios and the Xing MRI acquisition metadata.
- **recommended**: what section 7 of the atlas suggests adding: the data user guide, code list, data dictionary and LEDD method, the rest of the MDS-UPDRS, the medication log, diagnosis history, genetic status, biospecimen results, the Curated Data Cut, FreeSurfer and quality-control MRI tables, and image searches.
- **imaging**: every DaTscan and structural MRI table, and the raw DaTscan SPECT and structural MRI images: T1, T2, proton-density and FLAIR series, 2D and 3D, for every research group and visit.

In [ ]:
p.sql("""SELECT project, "group", subgroup, dataset, format, notes FROM dictionary WHERE project <> '' ORDER BY project, "group", dataset""")

PPMI's *Data Dictionary (Annotated)*, in the recommended set, describes every column of every table. Once you have it, `dictionary_columns` lines its descriptions up with the columns of the tables you loaded: `loaded` says whether your download has the column.

In [ ]:
p.sql("""
    SELECT "table", count(*) AS columns, count(*) FILTER (description <> '') AS described, count(*) FILTER (loaded) AS loaded
    FROM dictionary_columns GROUP BY "table" ORDER BY "table"
""")

In [ ]:
p.sql("""SELECT "column", description, loaded FROM dictionary_columns WHERE "table" = 'mds_updrs_part_iii' ORDER BY ALL LIMIT 20""")

## 2. What IDA offers

IDA's sign-in page asks for a human check, so code cannot sign in with your password, and IDA hands out no tokens. Instead you sign in in your browser and lend this notebook that session. It then makes the same requests as IDA's *Download Study Data* page here, and as its *Image Collections* page in section 6:

1. Sign in to IDA and open *Download* > *Study Data*. `ida.browse("study")` opens it.
2. Open the browser's developer tools (in Chrome: *View* > *Developer* > *Developer Tools*), choose *Network* and reload the page.
3. Right-click `studyData.jsp` in the list and choose *Copy* > *Copy as cURL*. Firefox and Safari have the same item.
4. Run the next cell and paste into the box. The box hides what you paste.

The session is your sign-in until IDA ends it. When it ends, the notebook says so: sign in again and copy the request again. It stays in this notebook's memory, is sent only to ida.loni.usc.edu, and is never printed. If you would rather not paste it each time, save it as `data/ida/session.txt`, which git ignores, and delete that file when you are done.

`ida.offer()` then opens each category of the Study Data page and saves the list of files as `data/ida/offer.csv`, so later runs show it without signing in; run it again to refresh it. Without a session, the notebook carries on with the last saved list and skips the downloads.

In [ ]:
try:
    display(ida.signin())
except (PermissionError, ValueError, ConnectionError, NotImplementedError) as error:
    print("Not signed in:", error or "this notebook cannot ask for input here.")

In [ ]:
if ida.signed_in():
    ida.offer()
p = data.PPMI(clean=False)
p.sql("""SELECT "group", count(*) AS files, count(DISTINCT subgroup) AS "sub-groups", max(updated) AS "newest version"
         FROM ida_files GROUP BY "group" ORDER BY "group" """)

`available` lines up the dictionary, IDA's list and the data folder, one row per dataset. `file_id` is IDA's ID for each file, `ida_category` where it sits on the Study Data page, and `downloaded` the file you have. A dataset IDA offers that the dictionary does not list keeps IDA's group, with the kind *other IDA file*.

In [ ]:
p.sql("""
    SELECT "group", count(*) AS datasets, count(*) FILTER (file_id <> '') AS "on IDA", count(*) FILTER (downloaded <> '') AS downloaded,
           count(*) FILTER (project <> '') AS "in the project's sets"
    FROM available GROUP BY "group" ORDER BY "group"
""")

In [ ]:
p.sql("""
    SELECT "group", subgroup, dataset, kind, project, ida_category, ida_updated, downloaded, release, "rows", participants
    FROM available ORDER BY "group", subgroup, dataset
""")

In [ ]:
plot.counts(p.sql("SELECT dataset, participants FROM available WHERE participants > 0 ORDER BY participants DESC, dataset"),
            "Participants in each download")

## 3. Choose

`wanted` is a SQL view of `available`. It starts as the project's four sets, plus every file IDA offers under *Imaging* > *DaTSCAN*, *MRI Acquisition* and *MR Analysis Results*, so every DaTscan and structural MRI table comes too. Change the condition in `WANTED` to choose differently: by group, sub-group, `ida_category`, kind or dataset name. Image collections are not on the Study Data page, so sections 5 and 6 get them.

In [ ]:
WANTED = """
    "group" <> 'Image Collections'
    AND (project <> ''
         OR regexp_matches(ida_category, 'Imaging > (DaTSCAN|MRI Acquisition|MR Analysis Results)'))
"""


def load(clean=False):
    """The data folder in DuckDB, with the files chosen above as the view wanted."""
    p = data.PPMI(clean=clean)
    p.db.execute(f"CREATE TEMP VIEW wanted AS SELECT * FROM available WHERE {WANTED}")
    return p


p = load()
p.sql("""
    SELECT coalesce(nullif(project, ''), 'DaTscan and MRI tables') AS "set", "group", dataset, ida_category, downloaded
    FROM wanted ORDER BY project = '', project, "group", dataset
""")

## 4. Download

`ida.download` asks IDA for every file in `wanted` that you do not have yet, ten at a time, and files what arrives: tables into `study/`, PDFs into `docs/`, zips unpacked and kept in `downloads/`. A dataset without a file ID is not on the Study Data page under that name. The genetic files, for example, are on IDA's *Genetic Data* page. Download those in the browser; `ida.collect()` then files them from your Downloads folder, or from the folder named by `IDA_DOWNLOADS`.

In [ ]:
missing = p.column("SELECT file_id FROM wanted WHERE downloaded = '' AND file_id <> '' ORDER BY dataset")
print(len(missing), "of the chosen datasets are on IDA and not downloaded yet.")
if missing and ida.signed_in():
    display(ida.download(missing))
display(ida.collect())
p = load()

In [ ]:
p.sql("""
    SELECT coalesce(nullif(project, ''), 'DaTscan and MRI tables') AS "set", "group", dataset,
           CASE WHEN file_id = '' THEN 'not on the Study Data page: download it in the browser' ELSE 'not downloaded yet' END AS why
    FROM wanted WHERE downloaded = '' ORDER BY project = '', project, "group", dataset
""")

## 5. Image searches

IDA's image search lists participants, so you run it in your browser; section 6 does everything after it. An image search lists one scan per row with its participant, research group, visit, modality, weighting and image ID. It ties scans to visits, `audit.ipynb` counts scans from it, and section 6 downloads the scans it found. In IDA's *Search*, choose *Advanced Search* and run two searches. Tick no research group and no visit, so that every participant and visit comes back:

| Search | Criteria | Collection |
|---|---|---|
| DaTscan | Modality *SPECT* | `PPMI DaTscan` |
| Structural MRI | Modality *MRI*, Weighting *T1*, *T2* and *PD* | `PPMI MRI` |

FLAIR series come with the T2 weighting, and both searches cover 2D and 3D acquisitions. In each search's results:

1. Show Subject ID, Research Group, Visit, Modality, Weighting, Acquisition Type, Image ID, Description and Study Date, and save the results with *CSV Download*. The CSV keeps only the columns shown, and it is the only list of the images that has their weighting.
2. Select every image and choose *Add To Collection*, naming a new collection as in the table. Section 6 downloads it.

Diffusion MRI is a third, optional search: Modality *DTI*.

The next cell files the CSVs from your Downloads folder into `imaging/`, where they become the `images` table, and lists your collections. If a search is still missing, it opens IDA's search page; save the searches and run the cell again.

In [ ]:
display(ida.collect())
if ida.signed_in():
    display(ida.collections())
p = load()
searched = set(p.column("SELECT DISTINCT modality FROM images"))
if not {"SPECT", "MRI"} <= searched:
    print("Still to search:", ", ".join(sorted({"SPECT", "MRI"} - searched)))
    ida.browse("search")
p.sql("""
    SELECT modality, coalesce(weighting, '') AS weighting, coalesce(acquisition, '') AS acquisition,
           count(DISTINCT "group") AS "research groups", count(DISTINCT event) AS visits,
           count(DISTINCT PATNO) AS participants, count(*) AS series
    FROM images GROUP BY ALL ORDER BY ALL
""")

## 6. Raw DaTscan and MRI images

`ida.images` downloads a collection with the session from section 2, as IDA's *1-Click Download* does with every image ticked:

- it saves the collection's list of images into `imaging/`, beside the search CSVs; an image on both lists counts once;
- it asks IDA for every image as archived, and keeps the links in `data/ida/`;
- it downloads the zips into `downloads/` and unpacks the images into `images/`, where they become the `image_files` table.

Run the cell again at any time: it resumes a broken transfer, skips zips it already has, and asks for new links when a collection has grown. If IDA has signed you out, sign in again in section 2. If the links have expired, `ida.images("PPMI MRI", refresh=True)` asks for new ones.

DaTscan for every participant and visit comes to tens of gigabytes. Structural MRI can run to hundreds. `ida.images("PPMI MRI", download=False)` lists the zips and their sizes without downloading, and `ida.images` stops before downloading when the disk lacks room for the zips and the unpacked images. `PPMI_DATA` can point the data folder at a larger disk.

A URL list saved from IDA's *Advanced Download* still works: save it into `data/ida/` with `urls` in its name, and the cell downloads it with `ida.fetch`.

In [ ]:
IMAGE_COLLECTIONS = ["PPMI DaTscan", "PPMI MRI"]

print(f"Free space on the data folder's disk: {shutil.disk_usage(p.root if p.root.exists() else Path.home()).free / 1e9:,.0f} GB")
if ida.signed_in():
    for name in IMAGE_COLLECTIONS:
        try:
            display(ida.images(name))
        except ValueError as error:
            print(error)
else:
    print("Sign in to IDA in section 2 to download the collections:", ", ".join(IMAGE_COLLECTIONS))
for url_list in sorted((p.root / "ida").glob("*urls*.txt")):
    print(url_list.name)
    display(ida.fetch(url_list))
p = load()

Every series on the image lists against the images you have, by image ID:

In [ ]:
p.sql(r"""
    WITH listed AS (
        SELECT modality, regexp_replace(trim(image_id), '^I|\.0+$', '', 'g') AS image_id FROM images WHERE image_id IS NOT NULL
    ), here AS (
        SELECT DISTINCT regexp_replace(image_id, '^I', '') AS image_id FROM image_files WHERE image_id IS NOT NULL
    )
    SELECT modality, count(DISTINCT l.image_id) AS "series listed", count(DISTINCT h.image_id) AS downloaded,
           count(DISTINCT l.image_id) - count(DISTINCT h.image_id) AS "still to download"
    FROM listed l LEFT JOIN here h USING (image_id) GROUP BY modality ORDER BY modality
""")

In [ ]:
p.sql("""
    SELECT format, count(*) AS series, count(DISTINCT PATNO) AS participants, round(sum(bytes) / 1e9, 2) AS gigabytes
    FROM image_files GROUP BY ALL ORDER BY format
""")

## 7. Check everything

This loads the data folder the way the other notebooks do, clean layer included. If a dataset you downloaded shows as missing, IDA named its file differently: it appears in `available` under *Other downloads*. Write its name without the date, such as `Participant_Status`, in the dataset's `file` column in `code/catalog.csv`.

In [ ]:
p = load(clean=True)
p.sql("""
    SELECT coalesce(nullif(project, ''), 'DaTscan and MRI tables') AS "set", count(*) AS datasets,
           count(*) FILTER (downloaded <> '') AS here, count(*) FILTER (downloaded = '') AS missing
    FROM wanted GROUP BY ALL ORDER BY "set"
""")

IDA updates tables between downloads, so tables downloaded on different days may disagree. For a final analysis, download every table on the same day.

In [ ]:
p.sql("""SELECT release AS "downloaded on", count(*) AS tables FROM downloads WHERE release IS NOT NULL GROUP BY ALL ORDER BY ALL""")

The clean layer reads its tables by name. `status` is `loaded` for each table it found and `missing` for the rest.

In [ ]:
if p.clean_error is not None:
    print(f"The clean layer was not built ({type(p.clean_error).__name__}). p.clean_error has the cause.")
if "clean.source_manifest" in p.tables():
    display(p.sql("SELECT source_key AS source, filename AS file, status, input_rows AS rows FROM clean.source_manifest ORDER BY status, source"))

## 8. Query everything with SQL

Reload after new downloads with `p = data.PPMI()`. One connection holds two layers.

- **main** has every downloaded table exactly as IDA ships it, named after its file in lower case: `Letter_-_Number_Sequencing_10Nov2026.csv` becomes `letter_number_sequencing`. Every column is text; the macros `num(x)` and `ppmi_date(x)` turn values into numbers and dates. `PATNO` and `EVENT_ID` are trimmed so that tables join cleanly. Beside them: `dictionary` and `dictionary_columns`, `ida_files`, `downloads` and `available` from the sections above, `images` (the image searches), `image_files` (the downloaded images) and the `cognition` view, which stacks the six cognitive tests.
- **clean** has the pipeline in `code/ppmi_pipeline`: typed and deduplicated tables keyed by `patient_id` and `event_id`, such as `clean.patients`, `clean.visits`, `clean.cognitive_assessments`, `clean.datscan_scans` and `clean.structural_scans`, with coverage views built on them.

`schema.ipynb` maps how all of these tables relate.

In [ ]:
p.sql("""
    SELECT CASE table_catalog WHEN 'memory' THEN 'main' ELSE table_catalog END AS layer, table_name AS "table", count(*) AS columns
    FROM information_schema.columns WHERE table_schema = 'main' AND table_catalog IN ('memory', 'clean')
    GROUP BY ALL ORDER BY layer, "table"
""")

In [ ]:
p.sql("""
    SELECT COHORT_DEFINITION AS cohort, lower(ENROLL_STATUS) AS status, count(*) AS participants
    FROM participant_status GROUP BY ALL ORDER BY cohort, status
""")

In [ ]:
p.sql("SELECT * FROM clean.availability_summary")

In [ ]:
plot.grouped(p.sql("""
    SELECT v.event_id,
           count(*) FILTER (has_six_tests) AS "all six tests",
           count(*) FILTER (has_datscan) AS "DaTscan",
           count(*) FILTER (has_structural) AS "grey-matter volume"
    FROM (VALUES ('SC'), ('BL'), ('V04'), ('V06'), ('V08'), ('V10'), ('V12')) v(event_id)
    LEFT JOIN clean.visit_coverage c USING (event_id)
    GROUP BY v.event_id ORDER BY visit_order(v.event_id)
"""), "Participants with each measurement, by visit")

### Outside Python

From the repository root:

```bash
python code/data.py                                         # what is loaded
python code/data.py "SELECT count(*) FROM clean.patients"
python code/data.py --export data/ppmi-2026-11-18.duckdb    # both layers in one DuckDB file, dated like the download
duckdb data/ppmi-2026-11-18.duckdb                          # query it with the DuckDB CLI or any SQL client
```

The exported file holds participant-level data. Keep it in `data/`, which git ignores, or outside the repository.

## Next

- `schema.ipynb` maps how the tables join.
- `data.ipynb` and `audit.ipynb` rerun the preliminary audit on this download.

Genetic status covers pathogenic variants such as LRRK2, GBA and SNCA, and APOE has its own table. MAPT H1 and COMT, which section 7 also names, need genotypes from IDA's *Genetic Data* page. Those files are large, so check their sizes there first.